# 01 — Bronze: ingestão dos snapshots do PI

Lê os CSVs depositados pelo Power Automate Desktop na biblioteca do SharePoint (trazidos ao OneLake pelo Dataflow Gen2) e grava tabelas Delta append-only, particionadas por `snapshot_date`.

**Princípios desta camada**

- **Nada é interpretado.** Todas as colunas entram como `string`. Tipagem é problema do silver.
- **Append-only.** Bronze nunca é alterado nem corrigido; é o arquivo permanente.
- **O manifesto manda.** Só entra dataset cuja execução reportou `success` ou `degraded`.
- **Idempotente.** Reprocessar a mesma pasta não duplica: `run_id` já ingerido é ignorado.

O manifesto também vira tabela (`bronze_run_manifest`), porque duração por dataset só serve para detectar degradação se virar série temporal.

In [ ]:
# Parâmetros — ajuste conforme o Lakehouse de destino
LANDING_PATH = "Files/landing/PI_System"   # onde o Dataflow Gen2 deposita as pastas datadas
BRONZE_PREFIX = "bronze_"
MANIFEST_TABLE = BRONZE_PREFIX + "run_manifest"

# Um dataset fora desta lista é ignorado com aviso, nunca ingerido às cegas.
DATASETS = [
    "pipoints", "pipoint_health", "digitalsets", "elements", "attributes",
    "templates", "eventframes", "analyses", "enumsets", "uom", "tables",
]

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException
import json


def table_exists(name):
    try:
        spark.table(name)
        return True
    except AnalysisException:
        return False


def read_inventory_csv(path):
    """Leitor único dos CSVs do PI.

    escape='"' é obrigatório: o gravador PowerShell usa RFC4180 (aspas duplicadas para escapar
    aspas) enquanto o padrão do Spark é barra invertida. Sem isso, campos com aspas desalinham
    as colunas silenciosamente.

    inferSchema fica desligado de propósito: bronze guarda texto cru.
    """
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("encoding", "UTF-8")
        .option("quote", '"')
        .option("escape", '"')
        .option("multiLine", "false")
        .csv(path)
    )

In [ ]:
manifest_glob = f"{LANDING_PATH}/*/_manifest__*.json"

try:
    manifest_rows = spark.read.option("multiLine", "true").json(manifest_glob).collect()
except AnalysisException:
    manifest_rows = []

print(f"Manifestos encontrados na landing: {len(manifest_rows)}")

if table_exists(MANIFEST_TABLE):
    already = {
        r["run_id"]
        for r in spark.table(MANIFEST_TABLE).select("run_id").distinct().collect()
    }
else:
    already = set()

pending = [r for r in manifest_rows if r["run_id"] not in already]
print(f"Já ingeridos: {len(already)} | Pendentes: {len(pending)}")

In [ ]:
ingested = []
skipped = []

for run in pending:
    run_id = run["run_id"]
    snapshot_date = run["snapshot_date"]

    for ds in run["datasets"]:
        name = ds["dataset"]
        status = ds["status"]
        file_name = ds["file"]

        if name not in DATASETS:
            skipped.append((run_id, name, "dataset desconhecido"))
            continue

        # Dataset que falhou na origem não entra: bronze não guarda dado incompleto.
        if status == "failed" or not file_name:
            skipped.append((run_id, name, f"status={status}"))
            continue

        path = f"{LANDING_PATH}/{snapshot_date}/{file_name}"
        try:
            df = read_inventory_csv(path)
        except AnalysisException:
            skipped.append((run_id, name, "arquivo ausente na landing"))
            continue

        df = (
            df.withColumn("_run_id", F.lit(run_id))
              .withColumn("_run_type", F.lit(run["run_type"]))
              .withColumn("_source_file", F.lit(file_name))
              .withColumn("_ingested_at", F.current_timestamp())
        )

        (df.write
           .format("delta")
           .mode("append")
           .partitionBy("snapshot_date")
           .option("mergeSchema", "true")
           .saveAsTable(BRONZE_PREFIX + name))

        ingested.append((run_id, name))

print(f"Datasets ingeridos: {len(ingested)} | ignorados: {len(skipped)}")
for s in skipped:
    print("  ignorado:", s)

In [ ]:
# O manifesto vira dataset: duração por dataset só detecta degradação como série temporal.
if pending:
    payloads = [json.dumps(r.asDict(recursive=True)) for r in pending]

    flat = (
        spark.read.json(spark.sparkContext.parallelize(payloads))
        .select(
            "run_id", "run_type", "snapshot_date", "snapshot_ts", "status",
            F.col("duration_ms").alias("run_duration_ms"),
            "module_version", "machine", "user",
            "pi_data_archive", "pi_data_archive_version",
            "af_server", "af_server_version", "af_database",
            "used_fallback_dir", "dataset_count", "failed_count", "degraded_count",
            "row_count_total",
            F.explode("datasets").alias("ds"),
        )
        .withColumn("dataset", F.col("ds.dataset"))
        .withColumn("dataset_status", F.col("ds.status"))
        .withColumn("dataset_row_count", F.col("ds.row_count"))
        .withColumn("dataset_duration_ms", F.col("ds.duration_ms"))
        .withColumn("dataset_file", F.col("ds.file"))
        .withColumn("dataset_error", F.col("ds.error"))
        .withColumn("dataset_metrics_json", F.to_json(F.col("ds.metrics")))
        .drop("ds")
        .withColumn("_ingested_at", F.current_timestamp())
    )

    (flat.write
        .format("delta")
        .mode("append")
        .partitionBy("snapshot_date")
        .option("mergeSchema", "true")
        .saveAsTable(MANIFEST_TABLE))

    print(f"{flat.count()} linhas de manifesto gravadas em {MANIFEST_TABLE}")
else:
    print("Nenhum manifesto novo.")

In [ ]:
for name in DATASETS:
    t = BRONZE_PREFIX + name
    if table_exists(t):
        df = spark.table(t)
        snaps = df.select("snapshot_date").distinct().count()
        print(f"{t:32} {df.count():>9,} linhas | {snaps:>4} snapshots")
    else:
        print(f"{t:32} (ainda não existe)")